# Sofiev Plume Rise Tuning with Physics-Guided Machine Learning

This notebook demonstrates how to use the refactored Sofiev plume rise tuning library. The original monolithic script has been broken down into a series of modules for improved maintainability and reusability.

In [ ]:
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import numpy as np
from src.gfs_ingestor import GFSIngestor
from src.satellite_ingestor import SyntheticIngestor
from src.sofiev_tuner import SofievTuner
from src.fortran_exporter import export_fortran_lut

## 1. Setup AWS GFS Connection

In [ ]:
gfs_handler = GFSIngestor()

## 2. Ingest Satellite Data

In [ ]:
sat_handler = SyntheticIngestor(n_samples=500)
start_time = datetime(2023, 7, 15, 12, 0)
end_time = start_time + timedelta(days=1)
bbox = (-125, 35, -100, 50)  # Western US

# Fetch data as an xarray.Dataset
ds_sat = sat_handler.fetch_data(start_time, end_time, bbox)

## 3. Collocate with GFS Data

In [ ]:
h_abl_list = []
n_ft_list = []
wind_list = []

# For demonstration, we simulate GFS data to avoid slow network calls.
# To use real GFS data, uncomment the loop below.
print("   (Simulating GFS values for speed...)")
h_abl_list = np.random.normal(1500, 400, ds_sat.event.size)
n_ft_list = np.abs(np.random.normal(0.012, 0.003, ds_sat.event.size))
wind_list = np.random.weibull(2, ds_sat.event.size) * 6

# --- REAL GFS FETCH LOOP (Uncomment for production) ---
# for i in range(ds_sat.event.size):
#     time = pd.to_datetime(ds_sat.time.values[i])
#     lat = ds_sat.lat.values[i]
#     lon = ds_sat.lon.values[i]
#     h_abl, wind, n_ft = gfs_handler.get_analysis_point(time, lat, lon)
#     h_abl_list.append(h_abl)
#     n_ft_list.append(n_ft)
#     wind_list.append(wind)

ds_collocated = ds_sat.assign(
    {
        "h_abl": (("event",), h_abl_list),
        "n_ft": (("event",), n_ft_list),
        "wind_speed": (("event",), wind_list),
    }
)

# Convert to pandas.DataFrame for compatibility with the tuner
df_raw = ds_collocated.to_dataframe()

## 4. Initialize Tuner

In [ ]:
tuner = SofievTuner()

## 5. Feature Engineering (PCA)

In [ ]:
df_proc = tuner.prepare_features(df_raw)

## 6. Train ML Model

In [ ]:
df_result = tuner.train(df_proc)

## 7. Generate Lookup Table for Fortran

In [ ]:
export_fortran_lut(tuner)

## 8. Validation Plot

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(
    df_result["frp"],
    df_result["target_beta"],
    c=df_result["wind_speed"],
    cmap="viridis",
    alpha=0.6,
)
plt.colorbar(label="Wind Speed (m/s)")
plt.xscale("log")
plt.xlabel("Fire Radiative Power (MW)")
plt.ylabel("Required Beta Parameter")
plt.title("Tuned Physics Parameter (Beta) by Fire Intensity & Wind")
plt.grid(True, alpha=0.3)
plt.show()